#  Context Management Strategy

#### Per Messages

In [3]:
import os
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv()

True

In [ ]:
# InMemorySaver persists the full message history per thread_id across invocations
agent = create_agent(
    model="gpt-4o-mini",
    checkpointer=InMemorySaver(),
    middleware=[SummarizationMiddleware(
        model="gpt-4o-mini",          # model used to write the summary, can differ from the agent model
        trigger=("messages", 10),     # summarize when the thread accumulates 10 messages
        keep=("messages", 4)          # after summarizing, keep the 4 most recent messages intact
    )],
)

In [ ]:
config = {
    "configurable": {
        "thread_id": "test-1"  # same thread_id = continuous conversation; history accumulates across calls
    }
}

questions = [
    "calc 1 + 1?",
    "calc 1 + 2?",
    "calc 1 + 3?",
    "calc 1 + 4?",
    "calc 1 + 5?",
    "calc 1 + 6?",
]

for q in questions:
    response = agent.invoke({ "messages": [HumanMessage(content=q)] }, config)
    print(response)
    # message count grows by 2 each turn (Q+A): 2 → 4 → 6 → 8 → 10
    # on the 6th question it drops to 6 because the middleware replaced older messages with a summary
    print(len(response["messages"]))

#### Per tokens

In [ ]:
def search_hotel(city: str) -> str:
    """ Search hotels - return long response to use more tokens """
    return f"""
    Hotels in {city}:
        1. Grand Hotel - 5 stars, 350$/night, spa, pool gym
        2. City Inn - 4 stars, 180$/night, business center
        3. Budget Stay - 3 stars, 75$/night, free wifi
"""

hotel_agent = create_agent(
    model="gpt-4o-mini",
    tools=[search_hotel],
    checkpointer=InMemorySaver(),
    middleware=[SummarizationMiddleware(
        model="gpt-4o-mini",
        trigger=("tokens", 500), # we can also use "fraction", 0.005
        keep=("tokens", 200)     # we can also use "fraction", 0.002
    )]
)

In [ ]:
config = {
    "configurable": {
        "thread_id": "test-1"  # same thread_id = continuous conversation; history accumulates across calls
    }
}

In [ ]:
cities = ["Paris", "France", "Italy", "United Kingdom"]

def count_tokens(messages):
    total_char = sum(len(str(m.content)) for m in messages)
    return total_char # chars approximate 1 token

for city in cities:
    response = hotel_agent.invoke({ "messages": [HumanMessage(content=f"find hotels in {city}")] }, config)

    tokens = ""
    print(f"{city} - ~{tokens} tokens - {len(response['messages'])} messages")
